# Genetic Disorder Classification — CatBoost Pipeline
10-class prediction, scored by Balanced Accuracy

In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q catboost scikit-learn imbalanced-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTENC

SEED = 42
np.random.seed(SEED)

In [3]:
# ── Cell 3: Load data ────────────────────────────────────────────────────────
train = pd.read_csv('train-data.csv')
label = pd.read_csv('train-label.csv')
test  = pd.read_csv('test-data.csv')

df = train.merge(label, on='id')
print(f'Train: {df.shape}, Test: {test.shape}')
print('Class distribution:')
print(df['disorder'].value_counts().sort_index())

Train: (13249, 43), Test: (8834, 42)
Class distribution:
disorder
0     389
1    2068
2    1090
3    3096
4      58
5    1700
6     813
7    2643
8      91
9    1301
Name: count, dtype: int64


In [4]:
# ── Cell 4: Preprocessing ────────────────────────────────────────────────────

# Columns to drop — zero variance or >70% missing with no class signal
DROP_COLS = [
    'id', 'first_name', 'last_name',
    'test_1', 'test_2', 'test_3', 'test_4', 'test_5',  # constant values
    'treatment_consent',                                  # always Y
    'autopsy',                                            # 70% missing, uniform across classes
    'insitute_name', 'institute_location',                # low signal, high cardinality
]

# Categorical columns (after dropping)
CAT_COLS = [
    'gender',
    'mother_defect', 'father_defect', 'maternal_defect', 'paternal_defect',
    'alive', 'respiration', 'heart_rate',
    'risk_level',
    'birth_asphyxia', 'place_birth', 'folic_acid',
    'maternal_illness', 'radiation_exposure', 'substance_abuse',
    'infertility_treatment', 'problem_previous_pregnancies',
    'birth_defects', 'blood_test',
    'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5',
]

# Numeric columns
NUM_COLS = ['age', 'blood_cell_count', 'mother_age', 'father_age',
            'abortion_cnt', 'white_blood_cell_count']


def preprocess(df, is_train=True):
    df = df.copy()

    # ── Feature engineering ──────────────────────────────────────────────────
    # Symptom count (most informative group)
    sym_cols = ['symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5']
    df['symptom_count'] = (df[sym_cols] == 'Y').sum(axis=1)

    # Defect count
    def_cols = ['mother_defect', 'father_defect', 'maternal_defect', 'paternal_defect']
    df['defect_count'] = (df[def_cols] == 'Y').sum(axis=1)

    # Parental age features
    df['parent_age_diff'] = df['father_age'] - df['mother_age']
    df['mother_old'] = (df['mother_age'] > 38).astype(float)

    # Row-level missingness (could correlate with data quality / disorder type)
    df['missing_count'] = df.isnull().sum(axis=1)

    # ── Numeric imputation ───────────────────────────────────────────────────
    for col in NUM_COLS + ['parent_age_diff']:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    # ── Categorical: fill NaN → 'MISSING' (CatBoost handles as category) ────
    for col in CAT_COLS:
        if col in df.columns:
            df[col] = df[col].fillna('MISSING').astype(str)

    # ── Drop useless columns ─────────────────────────────────────────────────
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

    if is_train:
        y = df.pop('disorder')
        return df, y
    return df


X, y = preprocess(df, is_train=True)
X_test = preprocess(test, is_train=False)

# Final categorical column indices for CatBoost
cat_features = [X.columns.get_loc(c) for c in CAT_COLS if c in X.columns]

print(f'X shape: {X.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Cat feature count: {len(cat_features)}')
print(f'Features: {list(X.columns)}')

X shape: (13249, 35)
X_test shape: (8834, 35)
Cat feature count: 24
Features: ['age', 'gender', 'mother_defect', 'father_defect', 'maternal_defect', 'paternal_defect', 'blood_cell_count', 'mother_age', 'father_age', 'alive', 'respiration', 'heart_rate', 'risk_level', 'birth_asphyxia', 'place_birth', 'folic_acid', 'maternal_illness', 'radiation_exposure', 'substance_abuse', 'infertility_treatment', 'problem_previous_pregnancies', 'abortion_cnt', 'birth_defects', 'white_blood_cell_count', 'blood_test', 'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5', 'symptom_count', 'defect_count', 'parent_age_diff', 'mother_old', 'missing_count']


In [5]:
# ── Cell 5: Class weights ────────────────────────────────────────────────────
classes = np.unique(y)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
class_weights = dict(zip(classes, weights))

print('Class weights (higher = rarer class gets more attention):')
for k, v in class_weights.items():
    print(f'  Class {k}: {v:.3f}  (n={sum(y==k)})')

Class weights (higher = rarer class gets more attention):
  Class 0: 3.406  (n=389)
  Class 1: 0.641  (n=2068)
  Class 2: 1.216  (n=1090)
  Class 3: 0.428  (n=3096)
  Class 4: 22.843  (n=58)
  Class 5: 0.779  (n=1700)
  Class 6: 1.630  (n=813)
  Class 7: 0.501  (n=2643)
  Class 8: 14.559  (n=91)
  Class 9: 1.018  (n=1301)


In [6]:
# ── Cell 6: CatBoost model definition ───────────────────────────────────────
model_params = dict(
    iterations        = 3000,
    learning_rate     = 0.05,
    depth             = 6,
    l2_leaf_reg       = 3,
    min_data_in_leaf  = 10,
    random_seed       = SEED,
    eval_metric       = 'TotalF1',     # proxy — we optimize BA at predict time
    loss_function     = 'MultiClass',
    class_weights     = list(weights), # balanced weights for rare classes
    early_stopping_rounds = 200,
    use_best_model    = True,
    verbose           = 200,
    task_type         = 'CPU',
)
print('Model params set.')

Model params set.


In [7]:
# ── Cell 7: Stratified K-Fold CV + OOF predictions ──────────────────────────
N_FOLDS = 10
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_preds  = np.zeros((len(X), 10))   # probability matrix OOF
test_preds = np.zeros((len(X_test), 10))
oof_ba_scores = []

X_arr = X.values
X_test_arr = X_test.values
y_arr = y.values

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_arr, y_arr)):
    print(f'\n======== Fold {fold+1}/{N_FOLDS} ========')

    X_tr, X_val = X_arr[trn_idx], X_arr[val_idx]
    y_tr, y_val = y_arr[trn_idx], y_arr[val_idx]

    train_pool = Pool(X_tr, label=y_tr, cat_features=cat_features)
    val_pool   = Pool(X_val, label=y_val, cat_features=cat_features)

    model = CatBoostClassifier(**model_params)
    model.fit(train_pool, eval_set=val_pool)

    fold_proba = model.predict_proba(val_pool)
    oof_preds[val_idx] = fold_proba

    fold_pred = np.argmax(fold_proba, axis=1)
    fold_ba   = balanced_accuracy_score(y_val, fold_pred)
    oof_ba_scores.append(fold_ba)
    print(f'Fold {fold+1} Balanced Accuracy: {fold_ba:.5f}')

    test_pool  = Pool(X_test_arr, cat_features=cat_features)
    test_preds += model.predict_proba(test_pool) / N_FOLDS

print('\n' + '='*50)
print(f'OOF Balanced Accuracy: {np.mean(oof_ba_scores):.5f} ± {np.std(oof_ba_scores):.5f}')
print(f'Per-fold scores: {[round(s,5) for s in oof_ba_scores]}')


======== Fold 1/10 ========
0:	learn: 0.2607089	test: 0.2257947	best: 0.2257947 (0)	total: 430ms	remaining: 21m 29s
200:	learn: 0.5329017	test: 0.3453071	best: 0.3790471 (77)	total: 1m 40s	remaining: 23m 24s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.3790471321
bestIteration = 77

Shrink model to first 78 iterations.
Fold 1 Balanced Accuracy: 0.38848

======== Fold 2/10 ========
0:	learn: 0.2795910	test: 0.2792251	best: 0.2792251 (0)	total: 423ms	remaining: 21m 7s
200:	learn: 0.5368303	test: 0.3354250	best: 0.3637671 (23)	total: 1m 37s	remaining: 22m 34s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.3637670853
bestIteration = 23

Shrink model to first 24 iterations.
Fold 2 Balanced Accuracy: 0.38602

======== Fold 3/10 ========
0:	learn: 0.2159480	test: 0.2005888	best: 0.2005888 (0)	total: 288ms	remaining: 14m 23s
200:	learn: 0.5323020	test: 0.3845980	best: 0.3873658 (118)	total: 1m 37s	remaining: 22m 43s
400:	learn: 0.6467068	test: 0.3

In [8]:
# ── Cell 8: OOF analysis — per-class accuracy ────────────────────────────────
from sklearn.metrics import classification_report

oof_labels = np.argmax(oof_preds, axis=1)
overall_ba = balanced_accuracy_score(y_arr, oof_labels)
print(f'Overall OOF Balanced Accuracy: {overall_ba:.5f}')

disorder_names = {
    0:'레베르시', 1:'낭포성섬유증', 2:'당뇨', 3:'리증후군', 4:'암',
    5:'테이-삭스', 6:'혈색소침착증', 7:'사립체근병종', 8:'알츠하이머', 9:'확인안됨'
}

print('\nPer-class recall (most important for Balanced Accuracy):')
report = classification_report(y_arr, oof_labels, output_dict=True)
for cls in range(10):
    r = report[str(cls)]['recall']
    n = sum(y_arr == cls)
    flag = ' ← LOW' if r < 0.3 else ''
    print(f'  Class {cls} ({disorder_names[cls]:12s}): recall={r:.3f}  n={n}{flag}')

Overall OOF Balanced Accuracy: 0.39915

Per-class recall (most important for Balanced Accuracy):
  Class 0 (레베르시        ): recall=0.404  n=389
  Class 1 (낭포성섬유증      ): recall=0.411  n=2068
  Class 2 (당뇨          ): recall=0.277  n=1090 ← LOW
  Class 3 (리증후군        ): recall=0.333  n=3096
  Class 4 (암           ): recall=0.707  n=58
  Class 5 (테이-삭스       ): recall=0.311  n=1700
  Class 6 (혈색소침착증      ): recall=0.534  n=813
  Class 7 (사립체근병종      ): recall=0.248  n=2643 ← LOW
  Class 8 (알츠하이머       ): recall=0.604  n=91
  Class 9 (확인안됨        ): recall=0.164  n=1301 ← LOW


In [ ]:
# ── Cell 9: Generate submission file ─────────────────────────────────────────
test_ids = test['id'].values
final_preds = np.argmax(test_preds, axis=1)

submission = pd.DataFrame({'id': test_ids, 'disorder': final_preds})
submission.to_csv('submission.csv', index=False)

print('submission.csv saved!')
print(f'Shape: {submission.shape}')
print('Prediction distribution:')
print(submission['disorder'].value_counts().sort_index())
print('\nFirst 5 rows:')
print(submission.head())

submissionb1.csv saved!
Shape: (8834, 2)
Prediction distribution:
disorder
0     454
1    1497
2     625
3    1549
4     177
5    1194
6    1037
7    1171
8     221
9     909
Name: count, dtype: int64

First 5 rows:
      id  disorder
0  16783         9
1  16784         6
2  16785         5
3  16786         1
4  16787         1


In [10]:
# ── Cell 10 (optional): Feature importance ───────────────────────────────────
# Uses the last fold's model — rough guide only
fi = pd.Series(
    model.get_feature_importance(),
    index=X.columns
).sort_values(ascending=False)

print('Top 20 features:')
print(fi.head(20).round(3).to_string())

Top 20 features:
symptom_5                       14.531
symptom_4                       12.767
symptom_count                   11.322
symptom_3                        9.471
defect_count                     8.939
symptom_2                        7.612
symptom_1                        6.520
father_defect                    4.587
maternal_defect                  3.631
missing_count                    3.347
mother_defect                    1.629
heart_rate                       1.082
paternal_defect                  0.993
problem_previous_pregnancies     0.989
folic_acid                       0.932
infertility_treatment            0.908
blood_test                       0.899
abortion_cnt                     0.802
radiation_exposure               0.794
gender                           0.791
